In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ball_detection").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "ball_detection").exists():
    raise RuntimeError(f"Could not find project root from {Path.cwd().resolve()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


PosixPath('/home/skynet/projects/volleyball/hawk-eye-volley')

In [2]:
import torch

from ball_detection.data.npz_dataset import (
    build_dataloaders,
    collect_train_val_npz_paths,
    get_input_channels,
)
from ball_detection.training.train_classifier import (
    BallCandidateCNN,
    fit,
    run_dataset_sanity_checks,
)


/home/skynet/projects/volleyball/hawk-eye-volley/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
INPUT_MODE = "rgb_brightness"
TRAIN_BALL_DIR = Path("/home/skynet/Downloads/hawk eye/data/_ball/npz/train")
VAL_BALL_DIR = Path("/home/skynet/Downloads/hawk eye/data/_ball/npz/val")
TRAIN_NO_BALL_DIR = Path("/home/skynet/Downloads/hawk eye/data/_ball not/npz/train")
VAL_NO_BALL_DIR = Path("/home/skynet/Downloads/hawk eye/data/_ball not/npz/val")

BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = MODELS_DIR / "ball_cnn_best.pt"


In [4]:
train_split, val_split = collect_train_val_npz_paths(
    train_ball_dir=TRAIN_BALL_DIR,
    val_ball_dir=VAL_BALL_DIR,
    train_no_ball_dir=TRAIN_NO_BALL_DIR,
    val_no_ball_dir=VAL_NO_BALL_DIR,
)

sanity_summary = run_dataset_sanity_checks(
    train_split=train_split,
    val_split=val_split,
    input_mode=INPUT_MODE,
)
sanity_summary


train: ball=48 no_ball=354 total=402
val: ball=22 no_ball=137 total=159
Sanity checks passed:
  train_ball_count: 48
  train_no_ball_count: 354
  val_ball_count: 22
  val_no_ball_count: 137
  input_mode: rgb_brightness
  in_channels: 4
  sample_shape: (4, 64, 64)


{'train_ball_count': 48,
 'train_no_ball_count': 354,
 'val_ball_count': 22,
 'val_no_ball_count': 137,
 'input_mode': 'rgb_brightness',
 'in_channels': 4,
 'sample_shape': (4, 64, 64)}

In [ ]:
train_loader, val_loader = build_dataloaders(
    train_paths=train_split.all_paths,
    val_paths=val_split.all_paths,
    input_mode=INPUT_MODE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

in_channels = get_input_channels(INPUT_MODE)

# Replace this with your own model if you already have one.
model = BallCandidateCNN(in_channels=in_channels)

sample_x, sample_y = next(iter(train_loader))
sample_batch_size = min(sample_x.shape[0], 4)
sample_logits = model(sample_x[:sample_batch_size])
sample_loss = torch.nn.BCEWithLogitsLoss()(
    sample_logits.reshape(-1),
    sample_y[:sample_batch_size].reshape(-1),
)

print(f"sample_x shape: {tuple(sample_x.shape)}")
print(f"sample_y shape: {tuple(sample_y.shape)}")
print(f"sample_logits shape: {tuple(sample_logits.shape)}")
print(f"sample_loss: {sample_loss.item():.4f}")


sample_x shape: (64, 4, 64, 64)
sample_y shape: (64,)
sample_logits shape: (4, 1)
sample_loss: 0.6307


In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    threshold=THRESHOLD,
    model_save_path=MODEL_SAVE_PATH,
)

history[-1]


Using device: cuda
Epoch 001/020 train_loss=0.5474 val_loss=0.6299 val_acc=0.8616 val_precision=0.0000 val_recall=0.0000 val_f1=0.0000 TP=0 TN=137 FP=0 FN=22
Saved best model to /home/skynet/projects/volleyball/hawk-eye-volley/ball_cnn_best.pt
Epoch 002/020 train_loss=0.3755 val_loss=0.5672 val_acc=0.8616 val_precision=0.0000 val_recall=0.0000 val_f1=0.0000 TP=0 TN=137 FP=0 FN=22
Saved best model to /home/skynet/projects/volleyball/hawk-eye-volley/ball_cnn_best.pt
Epoch 003/020 train_loss=0.2768 val_loss=0.5343 val_acc=0.9119 val_precision=0.6538 val_recall=0.7727 val_f1=0.7083 TP=17 TN=128 FP=9 FN=5
Saved best model to /home/skynet/projects/volleyball/hawk-eye-volley/ball_cnn_best.pt
Epoch 004/020 train_loss=0.2135 val_loss=0.5216 val_acc=0.8868 val_precision=0.5526 val_recall=0.9545 val_f1=0.7000 TP=21 TN=120 FP=17 FN=1
Epoch 005/020 train_loss=0.1639 val_loss=0.3261 val_acc=0.9371 val_precision=0.6875 val_recall=1.0000 val_f1=0.8148 TP=22 TN=127 FP=10 FN=0
Saved best model to /home/

{'epoch': 20,
 'train_loss': 0.04248875621428241,
 'val_loss': 0.10390906070458626,
 'val_accuracy': 0.9559748427672956,
 'val_precision': 0.9411764705882353,
 'val_recall': 0.7272727272727273,
 'val_f1': 0.8205128205128205,
 'tp': 16,
 'tn': 136,
 'fp': 1,
 'fn': 6}

In [ ]:
import matplotlib.pyplot as plt

from ball_detection.training.train_classifier import validate_one_epoch

checkpoint = torch.load(MODEL_SAVE_PATH, map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])

val_metrics = validate_one_epoch(
    model=model,
    dataloader=val_loader,
    criterion=torch.nn.BCEWithLogitsLoss(),
    threshold=THRESHOLD,
)

confusion_matrix = [
    [int(val_metrics["tn"]), int(val_metrics["fp"])],
    [int(val_metrics["fn"]), int(val_metrics["tp"])],
]
class_names = ["no_ball", "ball"]

fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(confusion_matrix, cmap="Blues")
fig.colorbar(image, ax=ax)

ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Validation Confusion Matrix")

max_count = max(max(row) for row in confusion_matrix)
for row_index, row in enumerate(confusion_matrix):
    for col_index, value in enumerate(row):
        text_color = "white" if value > max_count / 2 else "black"
        ax.text(
            col_index,
            row_index,
            str(value),
            ha="center",
            va="center",
            color=text_color,
            fontsize=12,
        )

plt.tight_layout()
plt.show()

val_metrics
